E-7. Regressione logistica e random forest sulle 48 statistiche, con il protocollo del §4.4, per K = 2 e K = 20. Riportiamo F1 pesato e matrice di
confusione.

In [ ]:
import pandas as pd
import numpy as np

E6_PATH = (
    "/content/drive/MyDrive/Neural Collapse/"
    "USTC-TFC2016/processed/ustc_tfc2016_preprocessed.parquet"
)

print("Caricamento dataset E-6...")

df = pd.read_parquet(E6_PATH)

print("Dataset caricato!")
print("Flussi:", len(df))
print("Colonne:", len(df.columns))

print("\nDistribuzione split:")
print(df["split"].value_counts())

print("\nClassi label_20:", df["label_20"].nunique())
print("Classi label_2:", df["label_2"].nunique())

Caricamento dataset E-6...
Dataset caricato!
Flussi: 562549
Colonne: 66

Distribuzione split:
split
train    393783
val       84383
test      56255
probe     28128
Name: count, dtype: int64

Classi label_20: 20
Classi label_2: 2


In [3]:
df_final = df

print("df_final pronto:", df_final.shape)

df_final pronto: (562549, 66)


In [4]:
# E-7 — Creazione delle 48 statistiche aggregate

import numpy as np

print("Creazione matrice delle 48 statistiche...")

stats_48 = np.column_stack([

    # DURATA / CONTEGGI / BYTES — 9
    df_final["bidirectional_duration_ms"].to_numpy(),
    df_final["bidirectional_packets"].to_numpy(),
    df_final["bidirectional_bytes"].to_numpy(),

    df_final["src2dst_duration_ms"].to_numpy(),
    df_final["src2dst_packets"].to_numpy(),
    df_final["src2dst_bytes"].to_numpy(),

    df_final["dst2src_duration_ms"].to_numpy(),
    df_final["dst2src_packets"].to_numpy(),
    df_final["dst2src_bytes"].to_numpy(),

    # PACKET SIZE — 12
    df_final["bidirectional_min_ps"].to_numpy(),
    df_final["bidirectional_mean_ps"].to_numpy(),
    df_final["bidirectional_stddev_ps"].to_numpy(),
    df_final["bidirectional_max_ps"].to_numpy(),

    df_final["src2dst_min_ps"].to_numpy(),
    df_final["src2dst_mean_ps"].to_numpy(),
    df_final["src2dst_stddev_ps"].to_numpy(),
    df_final["src2dst_max_ps"].to_numpy(),

    df_final["dst2src_min_ps"].to_numpy(),
    df_final["dst2src_mean_ps"].to_numpy(),
    df_final["dst2src_stddev_ps"].to_numpy(),
    df_final["dst2src_max_ps"].to_numpy(),

    # PIAT — 12
    df_final["bidirectional_min_piat_ms"].to_numpy(),
    df_final["bidirectional_mean_piat_ms"].to_numpy(),
    df_final["bidirectional_stddev_piat_ms"].to_numpy(),
    df_final["bidirectional_max_piat_ms"].to_numpy(),

    df_final["src2dst_min_piat_ms"].to_numpy(),
    df_final["src2dst_mean_piat_ms"].to_numpy(),
    df_final["src2dst_stddev_piat_ms"].to_numpy(),
    df_final["src2dst_max_piat_ms"].to_numpy(),

    df_final["dst2src_min_piat_ms"].to_numpy(),
    df_final["dst2src_mean_piat_ms"].to_numpy(),
    df_final["dst2src_stddev_piat_ms"].to_numpy(),
    df_final["dst2src_max_piat_ms"].to_numpy(),

    # FLAG TCP — 15
    df_final["bidirectional_syn_packets"].to_numpy(),
    df_final["bidirectional_cwr_packets"].to_numpy(),
    df_final["bidirectional_ece_packets"].to_numpy(),
    df_final["bidirectional_urg_packets"].to_numpy(),
    df_final["bidirectional_ack_packets"].to_numpy(),
    df_final["bidirectional_psh_packets"].to_numpy(),
    df_final["bidirectional_rst_packets"].to_numpy(),
    df_final["bidirectional_fin_packets"].to_numpy(),

    df_final["src2dst_syn_packets"].to_numpy(),
    df_final["src2dst_ack_packets"].to_numpy(),
    df_final["src2dst_psh_packets"].to_numpy(),
    df_final["src2dst_rst_packets"].to_numpy(),

    df_final["dst2src_syn_packets"].to_numpy(),
    df_final["dst2src_ack_packets"].to_numpy(),
    df_final["dst2src_psh_packets"].to_numpy()
])

print("Matrice creata!")
print("Dimensioni:", stats_48.shape)

print("\nControllo NaN:", np.isnan(stats_48).sum())
print("Controllo inf:", np.isinf(stats_48).sum())

Creazione matrice delle 48 statistiche...
Matrice creata!
Dimensioni: (562549, 48)

Controllo NaN: 0
Controllo inf: 0


In [5]:
# PREPARAZIONE DATI E-7 — K = 2

import numpy as np

print("## Preparazione dati K = 2...")

# Le 48 statistiche calcolate sopra
X = stats_48.astype(np.float32)

# Target binario
y = df["label_2"].to_numpy()

# Split già definiti in E-6: NON vengono modificati
split = df["split"].to_numpy()

# Maschere per i quattro split
train_mask = split == "train"
val_mask   = split == "val"
test_mask  = split == "test"
probe_mask = split == "probe"

# Creazione dei dataset
X_train = X[train_mask]
y_train = y[train_mask]

X_val = X[val_mask]
y_val = y[val_mask]

X_test = X[test_mask]
y_test = y[test_mask]

X_probe = X[probe_mask]
y_probe = y[probe_mask]

print("\nDimensioni:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Probe:", X_probe.shape)

print("\nNumero classi:", len(np.unique(y_train)))

print("\nPreparazione K = 2 completata.")

## Preparazione dati K = 2...

Dimensioni:
Train: (393783, 48)
Validation: (84383, 48)
Test: (56255, 48)
Probe: (28128, 48)

Numero classi: 2

Preparazione K = 2 completata.


In [ ]:
# E-7 — STANDARDIZZAZIONE + REGRESSIONE LOGISTICA K = 2

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

print("Standardizzazione delle 48 statistiche...")

# Lo scaler viene FITTATO SOLO sul TRAIN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Validation, test e probe vengono SOLO trasformati
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
X_probe_scaled = scaler.transform(X_probe)

print("Standardizzazione completata.")

print("\nAddestramento Regressione Logistica K = 2...")

lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr.fit(X_train_scaled, y_train)

# Validation
y_val_pred = lr.predict(X_val_scaled)
f1_val = f1_score(
    y_val,
    y_val_pred,
    average="weighted"
)

# Test
y_test_pred = lr.predict(X_test_scaled)
f1_test = f1_score(
    y_test,
    y_test_pred,
    average="weighted"
)

cm_test = confusion_matrix(y_test, y_test_pred)

print("\nREGRESSIONE LOGISTICA K = 2")
print("F1 weighted validation:", f1_val)
print("F1 weighted test:", f1_test)

print("\nMatrice di confusione TEST:")
print(cm_test)

Standardizzazione delle 48 statistiche...
Standardizzazione completata.

Addestramento Regressione Logistica K = 2...

REGRESSIONE LOGISTICA K = 2
F1 weighted validation: 0.9930409256573689
F1 weighted test: 0.9934205227454526

Matrice di confusione TEST:
[[30896    93]
 [  277 24989]]


In [6]:
# E-7 — RANDOM FOREST K = 2

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

print("Addestramento Random Forest K = 2...")

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Validation
y_val_pred_rf = rf.predict(X_val)

f1_val_rf = f1_score(
    y_val,
    y_val_pred_rf,
    average="weighted"
)

# Test
y_test_pred_rf = rf.predict(X_test)

f1_test_rf = f1_score(
    y_test,
    y_test_pred_rf,
    average="weighted"
)

cm_test_rf = confusion_matrix(
    y_test,
    y_test_pred_rf
)

print("\nRANDOM FOREST K = 2")
print("F1 weighted validation:", f1_val_rf)
print("F1 weighted test:", f1_test_rf)

print("\nMatrice di confusione TEST:")
print(cm_test_rf)

Addestramento Random Forest K = 2...

RANDOM FOREST K = 2
F1 weighted validation: 0.999976298601282
F1 weighted test: 1.0

Matrice di confusione TEST:
[[30989     0]
 [    0 25266]]


7-E bis:

In [7]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

# Stesso ordine ESATTO con cui ha impilato le colonne in stats_48 (E-7).
feature_names = [
    "bidirectional_duration_ms", "bidirectional_packets", "bidirectional_bytes",
    "src2dst_duration_ms", "src2dst_packets", "src2dst_bytes",
    "dst2src_duration_ms", "dst2src_packets", "dst2src_bytes",

    "bidirectional_min_ps", "bidirectional_mean_ps", "bidirectional_stddev_ps",
    "bidirectional_max_ps",

    "src2dst_min_ps", "src2dst_mean_ps", "src2dst_stddev_ps", "src2dst_max_ps",

    "dst2src_min_ps", "dst2src_mean_ps", "dst2src_stddev_ps", "dst2src_max_ps",

    "bidirectional_min_piat_ms", "bidirectional_mean_piat_ms",
    "bidirectional_stddev_piat_ms", "bidirectional_max_piat_ms",

    "src2dst_min_piat_ms", "src2dst_mean_piat_ms",
    "src2dst_stddev_piat_ms", "src2dst_max_piat_ms",

    "dst2src_min_piat_ms", "dst2src_mean_piat_ms",
    "dst2src_stddev_piat_ms", "dst2src_max_piat_ms",

    "bidirectional_syn_packets", "bidirectional_cwr_packets",
    "bidirectional_ece_packets", "bidirectional_urg_packets",

    "bidirectional_ack_packets", "bidirectional_psh_packets",
    "bidirectional_rst_packets", "bidirectional_fin_packets",

    "src2dst_syn_packets", "src2dst_ack_packets",
    "src2dst_psh_packets", "src2dst_rst_packets",

    "dst2src_syn_packets", "dst2src_ack_packets",
    "dst2src_psh_packets",
]

assert len(feature_names) == 48 == X_train.shape[1]


# ============================================================
# 1) IMPORTANZA DA IMPURITÀ
# ============================================================

imp_impurity = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)


# ============================================================
# 2) PERMUTATION IMPORTANCE
# ============================================================

perm_result = permutation_importance(
    rf,
    X_val,
    y_val,
    scoring="f1_weighted",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

imp_perm = pd.Series(
    perm_result.importances_mean,
    index=feature_names
).sort_values(ascending=False)


print("=== TOP 15 — importanza da impurità (K=2) ===")
print(imp_impurity.head(15))

print()

print("=== TOP 15 — permutation importance su validation (K=2) ===")
print(imp_perm.head(15))


# ============================================================
# 3) DIVISIONE DELLE FEATURE IN DUE GRUPPI
# ============================================================

# Feature di scala grezza:
# durata, numero totale di pacchetti e byte totali del flusso
raw_scale = [
    "bidirectional_duration_ms",
    "bidirectional_packets",
    "bidirectional_bytes",
    "src2dst_duration_ms",
    "src2dst_packets",
    "src2dst_bytes",
    "dst2src_duration_ms",
    "dst2src_packets",
    "dst2src_bytes",
]

# Feature di forma:
# statistiche delle dimensioni dei pacchetti,
# statistiche degli inter-arrivi (PIAT)
# e flag TCP
shape_like = [
    "bidirectional_min_ps",
    "bidirectional_mean_ps",
    "bidirectional_stddev_ps",
    "bidirectional_max_ps",

    "src2dst_min_ps",
    "src2dst_mean_ps",
    "src2dst_stddev_ps",
    "src2dst_max_ps",

    "dst2src_min_ps",
    "dst2src_mean_ps",
    "dst2src_stddev_ps",
    "dst2src_max_ps",

    "bidirectional_min_piat_ms",
    "bidirectional_mean_piat_ms",
    "bidirectional_stddev_piat_ms",
    "bidirectional_max_piat_ms",

    "src2dst_min_piat_ms",
    "src2dst_mean_piat_ms",
    "src2dst_stddev_piat_ms",
    "src2dst_max_piat_ms",

    "dst2src_min_piat_ms",
    "dst2src_mean_piat_ms",
    "dst2src_stddev_piat_ms",
    "dst2src_max_piat_ms",

    "bidirectional_syn_packets",
    "bidirectional_cwr_packets",
    "bidirectional_ece_packets",
    "bidirectional_urg_packets",

    "bidirectional_ack_packets",
    "bidirectional_psh_packets",
    "bidirectional_rst_packets",
    "bidirectional_fin_packets",

    "src2dst_syn_packets",
    "src2dst_ack_packets",
    "src2dst_psh_packets",
    "src2dst_rst_packets",

    "dst2src_syn_packets",
    "dst2src_ack_packets",
    "dst2src_psh_packets",
]


# ============================================================
# 4) CONTROLLI RICHIESTI DALLA TRACCIA
# ============================================================

assert set(raw_scale) | set(shape_like) == set(feature_names)

assert set(raw_scale) & set(shape_like) == set()


print()

print(
    "Somma importanza (impurità) — feature di scala grezza:",
    imp_impurity[raw_scale].sum()
)

print(
    "Somma importanza (impurità) — feature di forma:",
    imp_impurity[shape_like].sum()
)

print()

print(
    "Somma totale importanze:",
    imp_impurity.sum()
)

=== TOP 15 — importanza da impurità (K=2) ===
dst2src_min_ps                  0.127619
bidirectional_mean_piat_ms      0.094841
src2dst_duration_ms             0.088947
bidirectional_duration_ms       0.080651
bidirectional_max_piat_ms       0.071715
bidirectional_min_ps            0.065416
src2dst_min_ps                  0.046299
src2dst_stddev_piat_ms          0.044056
src2dst_mean_piat_ms            0.042712
dst2src_mean_ps                 0.038225
bidirectional_stddev_piat_ms    0.035439
src2dst_max_piat_ms             0.031831
bidirectional_ack_packets       0.030863
src2dst_ack_packets             0.030701
dst2src_ack_packets             0.021638
dtype: float64

=== TOP 15 — permutation importance su validation (K=2) ===
bidirectional_min_ps         0.000872
src2dst_ack_packets          0.000408
bidirectional_ack_packets    0.000014
bidirectional_duration_ms    0.000000
src2dst_packets              0.000000
src2dst_bytes                0.000000
bidirectional_bytes          0.0000

In [ ]:
# PREPARAZIONE DATI K = 20

import numpy as np

print("## Preparazione dati K = 20...")

# Feature: 48 statistiche
# stats_48 è già un numpy.ndarray

X = stats_48.astype(np.float32)

# Target: classificazione a 20 classi

y = df["label_20"].values

# Split già definiti in E-6
# NON vengono modificati

train_mask = df["split"].values == "train"
val_mask   = df["split"].values == "val"
test_mask  = df["split"].values == "test"
probe_mask = df["split"].values == "probe"

X_train = X[train_mask]
X_val   = X[val_mask]
X_test  = X[test_mask]
X_probe = X[probe_mask]

y_train = y[train_mask]
y_val   = y[val_mask]
y_test  = y[test_mask]
y_probe = y[probe_mask]

print()
print("Dimensioni:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Probe:", X_probe.shape)

print()
print("Numero classi:", len(np.unique(y_train)))

print()
print("Preparazione K = 20 completata.")

## Preparazione dati K = 20...

Dimensioni:
Train: (393783, 48)
Validation: (84383, 48)
Test: (56255, 48)
Probe: (28128, 48)

Numero classi: 20

Preparazione K = 20 completata.


In [ ]:
# REGRESSIONE LOGISTICA — K = 20

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

print("Addestramento Regressione Logistica K = 20...")

lr_20 = LogisticRegression(
    max_iter=500,
    random_state=42,
    n_jobs=-1
)

lr_20.fit(X_train, y_train)

# Validation
y_val_pred = lr_20.predict(X_val)
f1_val = f1_score(y_val, y_val_pred, average="weighted")

# Test
y_test_pred = lr_20.predict(X_test)
f1_test = f1_score(y_test, y_test_pred, average="weighted")

cm_test = confusion_matrix(y_test, y_test_pred)

print()
print("REGRESSIONE LOGISTICA K = 20")
print("F1 weighted validation:", f1_val)
print("F1 weighted test:", f1_test)

print()
print("Matrice di confusione TEST:")
print(cm_test)

Addestramento Regressione Logistica K = 20...

REGRESSIONE LOGISTICA K = 20
F1 weighted validation: 0.4671231065704996
F1 weighted test: 0.4661355847820558

Matrice di confusione TEST:
[[   0    0    0    0    0    0    0    0  297    0    0  455    0    0
     0    0    0    0    0    0]
 [   0 2970    0    0 3192    0    2    1    1    4    0    0    0    0
     0    0    0    0    0    0]
 [   0   93 4798    0    0    5    0    0 5094    0    0    5    0    0
     0    0  109    0    0    0]
 [   0    0    0    0    0    0    0    0  600    0    0    0    0    0
     0    0    0    0    0    0]
 [   0   12   17    0 4302    0  106   71  473   10   54    0    0    0
     0    0    1    0    0    0]
 [   0    1  259    0    0  147    0    0   89    0    0  367    0    0
     0    0    0    0    0    0]
 [   0   72   33    0   99    0  168   79  182  101    8    0    0    1
     0    4   54    0    3    7]
 [   0    4  309    0  733    1   11  372   17    0   24    0    0    0
     0  

In [ ]:

# RANDOM FOREST — K = 20

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

print("Addestramento Random Forest K = 20...")

rf_20 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_20.fit(X_train, y_train)

# Validation
y_val_pred_rf = rf_20.predict(X_val)
f1_val_rf = f1_score(y_val, y_val_pred_rf, average="weighted")

# Test
y_test_pred_rf = rf_20.predict(X_test)
f1_test_rf = f1_score(y_test, y_test_pred_rf, average="weighted")

cm_test_rf = confusion_matrix(y_test, y_test_pred_rf)

print()
print("RANDOM FOREST K = 20")
print("F1 weighted validation:", f1_val_rf)
print("F1 weighted test:", f1_test_rf)

print()
print("Matrice di confusione TEST:")
print(cm_test_rf)

Addestramento Random Forest K = 20...

RANDOM FOREST K = 20
F1 weighted validation: 0.8527825262916349
F1 weighted test: 0.8551606401352666

Matrice di confusione TEST:
[[ 478    0    0    0    0    0    0    0    0    0    0  273    0    0
     1    0    0    0    0    0]
 [   0 4840    0    0 1297    0    1   31    0    0    0    0    0    1
     0    0    0    0    0    0]
 [   0    0 9889    0    0    0    0    0  215    0    0    0    0    0
     0    0    0    0    0    0]
 [   0    0    0  600    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [   0  629    0    0 4199    0   11  204    0    0    0    0    0    3
     0    0    0    0    0    0]
 [   4    0  202    0    0  596    0    0    0    0    0   56    0    0
     4    0    0    0    1    0]
 [   0    3    0    0   99    0  705    0    0    1    1    0    0    1
     0    1    0    0    0    0]
 [   0   36    0    0  504    0    2  927    0    0    2    0    0    0
     0    0    0    0   

In [ ]:
import pandas as pd

# SALVATAGGIO RISULTATI E-7

risultati_e7 = pd.DataFrame({
    "K": [2, 2, 20, 20],
    "Modello": [
        "Regressione Logistica",
        "Random Forest",
        "Regressione Logistica",
        "Random Forest"
    ],
    "F1_validation": [
        0.9930409256573689,
        0.999976298601282,
        0.4671231065704996,
        0.8527825262916349
    ],
    "F1_test": [
        0.9934205227454526,
        1.0,
        0.4661355847820558,
        0.8551606401352666
    ]
})

# Salva il file
risultati_e7.to_csv("risultati_E7.csv", index=False)

print("====================================")
print("RISULTATI E-7 SALVATI")
print("====================================")
print(risultati_e7)
print()
print("File creato: risultati_E7.csv")

RISULTATI E-7 SALVATI
    K                Modello  F1_validation   F1_test
0   2  Regressione Logistica       0.993041  0.993421
1   2          Random Forest       0.999976  1.000000
2  20  Regressione Logistica       0.467123  0.466136
3  20          Random Forest       0.852783  0.855161

File creato: risultati_E7.csv
